# MoleculeNet benchmark: BACE, Clintox, ESOL (TEST replay)

This notebook re-runs **held-out TEST** evaluation in-process: `best_summary.json`, `src/main_arcmol.py`, flat **`model.pth`** (+ **`calib.json`** for classification), and **`eval_preprocess.pkl`** when `data/` only has `*_test.pkl`.

**Repository layout**
- `checkpoints/benchmark_{bace,clintox,esol}/` — `best_summary.json`, `model.pth`, `calib.json` (cls only), `eval_preprocess.pkl`.
- `data/{task}/` — public replay: **`{task}_test.pkl` only**; add `*_train.pkl` locally to recompute descriptors (artifact ignored).

**Paper-reported values** — BACE AUC **87.9** (%), Clintox **94.3** (%), ESOL RMSE **0.517**. Summary table below; parity uses `best_test` in each `best_summary.json`.


## Environment

PyTorch + sklearn; **CUDA recommended** for stable MC-dropout parity with `best_test`.


In [1]:
from __future__ import annotations

import io
import json
import os
import pickle
import re
import sys
from contextlib import redirect_stdout
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path("..").resolve()
os.chdir(REPO_ROOT)
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import main_arcmol as m

CHECKPOINTS = REPO_ROOT / "checkpoints"

PAPER = {
    "bace": {"display": "87.9 (AUC %)", "auc_0_1": 87.9 / 100.0},
    "clintox": {"display": "94.3 (AUC %)", "auc_0_1": 94.3 / 100.0},
    "esol": {"display": "0.517 (RMSE)", "rmse": 0.517},
}

assert CHECKPOINTS.is_dir(), f"Missing {CHECKPOINTS}"

TASKS = {
    "bace": CHECKPOINTS / "benchmark_bace" / "best_summary.json",
    "clintox": CHECKPOINTS / "benchmark_clintox" / "best_summary.json",
    "esol": CHECKPOINTS / "benchmark_esol" / "best_summary.json",
}

print("REPO_ROOT:", REPO_ROOT.resolve())
print("CHECKPOINTS:", CHECKPOINTS.resolve())
print("cwd:", os.getcwd())


REPO_ROOT: /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main
CHECKPOINTS: /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/checkpoints
cwd: /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main


In [2]:
def _apply_best_params(args, bp: dict) -> None:
    for k, v in bp.items():
        if not hasattr(args, k):
            continue
        if k == "moe_topk" and v is None:
            continue
        setattr(args, k, v)


def _resolve_paths(best_summary: Path, best_model_rel: str, summ: dict):
    task_root = best_summary.parent
    p = Path(best_model_rel)
    parent = p.parent
    parent_name = parent.name if str(parent) not in (".", "") else ""

    if parent_name.startswith("trial_"):
        trial_dir = task_root / parent_name
        ckpt = trial_dir / p.name
        stem = ckpt.stem
        prefix = "best_arcmol_arc_reg_"
        if stem.startswith(prefix):
            calib = trial_dir / f"calib_{stem[len(prefix):]}.json"
        else:
            calibs = sorted(trial_dir.glob("calib_*.json"))
            if not calibs:
                raise FileNotFoundError(f"No calib under {trial_dir}")
            calib = calibs[0]
        return ckpt, calib, trial_dir

    ckpt = (task_root / p).resolve()
    calib = None
    if summ.get("task_type") == "cls":
        rel = summ.get("calib_relpath")
        if isinstance(rel, str) and rel.strip():
            calib = (task_root / rel).resolve()
        else:
            calibs = sorted(task_root.glob("calib*.json"))
            if not calibs:
                raise FileNotFoundError(
                    f"Classification needs calib JSON in {task_root} or calib_relpath in summary"
                )
            calib = calibs[0]
    return ckpt, calib, task_root


def _artifact_path(summ_parent: Path, summ: dict) -> Path:
    rel = summ.get("eval_preprocess_relpath", "eval_preprocess.pkl")
    return (summ_parent / rel).resolve()


def compute_eval_preprocess_dict(args) -> dict[str, Any]:
    m.set_seed(args.seed)

    def _p(split: str) -> str:
        return os.path.join(args.data_dir, f"{args.task_name}_{split}.pkl")

    if not os.path.isfile(_p("train")):
        raise FileNotFoundError(_p("train"))
    if not os.path.isfile(_p("test")):
        raise FileNotFoundError(_p("test"))

    with open(_p("train"), "rb") as f:
        train_data = pickle.load(f)
    with open(_p("test"), "rb") as f:
        test_data = pickle.load(f)

    train_data = m.filter_data_for_training(
        train_data, args.fusion_embed_types, args.target_name
    )
    test_data = m.filter_data_for_training(
        test_data, args.fusion_embed_types, args.target_name
    )

    attributes = list(train_data[list(train_data.keys())[0]]["rdkit_descriptors"].keys())
    Xtr, ytr = m.extract_rdkit_and_target(
        train_data, attributes, args.target_name, task_type=args.task_type
    )
    Xte, _yte = m.extract_rdkit_and_target(
        test_data, attributes, args.target_name, task_type=args.task_type
    )
    Xtr_sel, topk = m.select_top_k_features(
        Xtr, ytr, k=args.k, task_type=args.task_type, random_state=args.seed
    )
    _ = Xte[:, topk]

    scaler = StandardScaler()
    scaler.fit(Xtr_sel)

    y_mu, y_sigma = 0.0, 1.0
    if args.task_type == "reg" and args.y_std_enable:
        emb_tr, y_tr = m.extract_selected_embedding(
            train_data, args.fusion_embed_types, args.target_name, task_type="reg"
        )
        y_mu = float(torch.mean(y_tr.float()).item())
        y_sigma = float(torch.std(y_tr.float()).item())
        y_sigma = y_sigma if y_sigma > args.y_std_eps else 1.0

    return {
        "topk": np.asarray(topk, dtype=np.int64),
        "scaler_mean": np.asarray(scaler.mean_, dtype=np.float64),
        "scaler_scale": np.asarray(scaler.scale_, dtype=np.float64),
        "attributes": attributes,
        "y_mu": y_mu,
        "y_sigma": y_sigma,
        "y_std_enable": bool(args.task_type == "reg" and args.y_std_enable),
    }


def export_eval_preprocess(summ_path: Path, out_name: str = "eval_preprocess.pkl") -> Path:
    summ_path = summ_path.resolve()
    with open(summ_path, encoding="utf-8") as f:
        summ = json.load(f)
    parser = m.build_arg_parser()
    base = parser.parse_args([])
    base.data_dir = summ["data_dir"]
    base.task_name = summ["task_name"]
    base.task_type = summ["task_type"]
    base.target_name = summ["target_name"]
    base.seed = int(summ.get("seed", 42)) if isinstance(summ.get("seed"), int) else 42
    _apply_best_params(base, summ["best_params"])
    art = compute_eval_preprocess_dict(base)
    out = summ_path.parent / out_name
    with open(out, "wb") as f:
        pickle.dump(art, f, protocol=pickle.HIGHEST_PROTOCOL)
    print("Wrote", out)
    return out


def eval_test_only(args, ckpt: Path, calib, summ_path: Path, summ: dict) -> float:
    m.set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] {device}")

    def _p(split: str) -> str:
        return os.path.join(args.data_dir, f"{args.task_name}_{split}.pkl")

    pre_path = _artifact_path(summ_path.parent, summ)
    train_missing = not os.path.isfile(_p("train"))
    use_artifact = train_missing and pre_path.is_file()

    if use_artifact:
        print(f"[Preprocess] using artifact {pre_path}")
        with open(pre_path, "rb") as f:
            art = pickle.load(f)
        if not os.path.isfile(_p("test")):
            raise FileNotFoundError(_p("test"))
        with open(_p("test"), "rb") as f:
            test_data = pickle.load(f)
        test_data = m.filter_data_for_training(
            test_data, args.fusion_embed_types, args.target_name
        )
        print(f"[After filter] train=(artifact) valid=(skip) test={len(test_data)}")

        attributes = list(art["attributes"])
        topk = art["topk"]
        Xte, yte = m.extract_rdkit_and_target(
            test_data, attributes, args.target_name, task_type=args.task_type
        )
        Xte_sel = Xte[:, topk]
        mean = art["scaler_mean"]
        scale = art["scaler_scale"]
        scale = np.where(scale < 1e-12, 1.0, scale)
        Xte_sel = (Xte_sel - mean) / scale
        Xte_sel = np.clip(Xte_sel, -1e6, 1e6)

        desc_te = torch.tensor(Xte_sel, dtype=torch.float32)
        emb_te, y_te = m.extract_selected_embedding(
            test_data, args.fusion_embed_types, args.target_name, task_type=args.task_type
        )

        y_mu, y_sigma = float(art.get("y_mu", 0.0)), float(art.get("y_sigma", 1.0))
        if art.get("y_std_enable"):
            y_te = (y_te - y_mu) / y_sigma
    else:
        for split in ("train", "test"):
            if not os.path.isfile(_p(split)):
                raise FileNotFoundError(_p(split))

        with open(_p("train"), "rb") as f:
            train_data = pickle.load(f)
        with open(_p("test"), "rb") as f:
            test_data = pickle.load(f)

        valid_exists = os.path.isfile(_p("valid"))
        if valid_exists:
            with open(_p("valid"), "rb") as f:
                valid_data = pickle.load(f)
            valid_data = m.filter_data_for_training(
                valid_data, args.fusion_embed_types, args.target_name
            )
        else:
            valid_data = {}

        train_data = m.filter_data_for_training(
            train_data, args.fusion_embed_types, args.target_name
        )
        test_data = m.filter_data_for_training(
            test_data, args.fusion_embed_types, args.target_name
        )

        n_valid = len(valid_data) if isinstance(valid_data, dict) else 0
        print(
            f"[After filter] train={len(train_data)} valid={n_valid} test={len(test_data)}"
        )

        attributes = list(train_data[list(train_data.keys())[0]]["rdkit_descriptors"].keys())
        Xtr, ytr = m.extract_rdkit_and_target(
            train_data, attributes, args.target_name, task_type=args.task_type
        )
        Xte, yte = m.extract_rdkit_and_target(
            test_data, attributes, args.target_name, task_type=args.task_type
        )
        Xtr_sel, topk = m.select_top_k_features(
            Xtr, ytr, k=args.k, task_type=args.task_type, random_state=args.seed
        )
        Xte_sel = Xte[:, topk]

        scaler = StandardScaler()
        Xtr_sel = scaler.fit_transform(Xtr_sel)
        Xte_sel = scaler.transform(Xte_sel)
        Xtr_sel = np.clip(Xtr_sel, -1e6, 1e6)
        Xte_sel = np.clip(Xte_sel, -1e6, 1e6)

        desc_te = torch.tensor(Xte_sel, dtype=torch.float32)
        emb_te, y_te = m.extract_selected_embedding(
            test_data, args.fusion_embed_types, args.target_name, task_type=args.task_type
        )

        y_mu, y_sigma = 0.0, 1.0
        if args.task_type == "reg" and args.y_std_enable:
            emb_tr, y_tr = m.extract_selected_embedding(
                train_data, args.fusion_embed_types, args.target_name, task_type="reg"
            )
            y_mu = float(torch.mean(y_tr.float()).item())
            y_sigma = float(torch.std(y_tr.float()).item())
            y_sigma = y_sigma if y_sigma > args.y_std_eps else 1.0
            y_te = (y_te - y_mu) / y_sigma

    test_loader = m.create_loader(emb_te, desc_te, y_te, bs=args.batch_size, shuffle=False)

    desc_module = m.TaskAwareDescriptorPooling(
        in_dim=desc_te.shape[1], h=128, out_dim=64, drop=0.1
    ).to(device)
    fusion_module = m.AttentionPoolingFusion(
        used_embedding_types=args.fusion_embed_types,
        l_output_dim=args.fusion_out_dim,
        hidden_dim=args.fusion_hidden_dim,
        dropout_prob=args.fusion_dropout,
        comp_mode=args.comp_mode,
        cka_gamma=args.cka_gamma,
        task_gate=args.task_gate,
        task_ctx_dim=args.task_ctx_dim,
        comp_scale=args.comp_scale,
        top_k=args.moe_topk,
        sparse_lambda=args.moe_sparse_lambda,
    ).to(device)

    in_dim = args.fusion_out_dim + 64
    num_classes = 2 if args.task_type == "cls" else 1
    model = m.ArcMolModel(
        fusion_module,
        desc_module,
        in_dim=in_dim,
        task_type=args.task_type,
        num_classes=num_classes,
        task_ctx_dim=args.task_ctx_dim,
        use_task_ctx=args.use_task_ctx,
        margin=args.margin,
        scale=args.scale,
        head_hidden=args.head_hidden,
        head_dropout=args.head_dropout,
        proxy_dropout=args.proxy_dropout,
        arc_reg_use=(args.arc_reg_use if args.task_type == "reg" else False),
        arc_reg_nbins=args.arc_reg_nbins,
        arc_reg_margin=args.arc_reg_margin,
        arc_reg_scale=args.arc_reg_scale,
        arc_reg_soft_sigma=args.arc_reg_soft_sigma,
    ).to(device)

    if not ckpt.is_file():
        raise FileNotFoundError(ckpt)
    state = torch.load(str(ckpt), map_location=device)
    model.load_state_dict(state, strict=True)
    model.eval()

    if args.task_type == "cls":
        if calib is None or not calib.is_file():
            raise FileNotFoundError(calib)
        with open(calib, "r", encoding="utf-8") as f:
            raw = json.load(f)
        T = float(raw.get("temperature", 1.0))
        thr = float(raw.get("prob_threshold", raw.get("threshold", 0.5)))
        print(f"[Calib] {calib.name}  T={T} thr={thr}")

        metrics, y, p, _ = m.eval_epoch(
            model,
            test_loader,
            device,
            task_type="cls",
            temperature=T,
            mc_passes=args.mc_passes,
        )
        from sklearn.metrics import f1_score

        f1 = f1_score(y, (p > thr).astype(int))
        cov = (p > thr).mean()
        acc = ((((p > thr).astype(int) == y) & (p > thr)).sum()) / max(int((p > thr).sum()), 1)
        print(f"\n=== TEST (CLS) ===")
        print(
            f"AUC: {metrics['AUC']:.6f} | F1@calib_thr: {f1:.6f} | "
            f"Coverage={cov:.4f}, Acc@accepted={acc:.4f}"
        )
        return float(metrics["AUC"])

    metrics, _, _, _ = m.eval_epoch(
        model,
        test_loader,
        device,
        task_type="reg",
        temperature=None,
        mc_passes=args.mc_passes,
        y_std_enable=args.y_std_enable,
        y_mu=y_mu,
        y_sigma=y_sigma,
    )
    print(f"\n=== TEST (REG) ===")
    print(f"RMSE: {metrics['RMSE']:.6f} | MAE: {metrics['MAE']:.6f}")
    return float(metrics["RMSE"])


def run_best_summary_eval(summ_path: Path) -> None:
    summ_path = summ_path.resolve()
    with open(summ_path, encoding="utf-8") as f:
        summ = json.load(f)
    bm = summ["best_model"]
    if not isinstance(bm, str):
        raise KeyError("best_model")
    parser = m.build_arg_parser()
    base = parser.parse_args([])
    base.data_dir = summ["data_dir"]
    base.task_name = summ["task_name"]
    base.task_type = summ["task_type"]
    base.target_name = summ["target_name"]
    base.seed = int(summ.get("seed", 42)) if isinstance(summ.get("seed"), int) else 42
    _apply_best_params(base, summ["best_params"])

    ckpt, calib, run_root = _resolve_paths(summ_path, bm, summ)
    print(f"[Summary] {summ_path}")
    print(f"[Ckpt]    {ckpt}")
    print(f"[Reported] metric={summ.get('metric')} best_test={summ.get('best_test')}")

    metric = eval_test_only(base, ckpt, calib, summ_path, summ)
    rep = summ.get("best_test")
    if rep is not None:
        if summ["task_type"] == "cls":
            ok = abs(float(metric) - float(rep)) < 1e-4
        else:
            ok = abs(float(metric) - float(rep)) < 1e-3
        print(f"\n[Check] reproduced={metric:.6f} vs reported={float(rep):.6f} -> {'OK' if ok else 'MISMATCH'}")


def run_eval(best_summary: Path) -> str:
    buf = io.StringIO()
    with redirect_stdout(buf):
        run_best_summary_eval(Path(best_summary))
    return buf.getvalue()


def parse_log(log: str, task_type: str) -> dict:
    d: dict = {}
    if task_type == "cls":
        r = re.search(r"AUC:\s*([0-9.]+)", log)
        if r:
            d["AUC"] = float(r.group(1))
        r2 = re.search(r"F1@calib_thr:\s*([0-9.]+)", log)
        if r2:
            d["F1_calib"] = float(r2.group(1))
    else:
        r = re.search(r"RMSE:\s*([0-9.]+)\s*\|\s*MAE:\s*([0-9.]+)", log)
        if r:
            d["RMSE"] = float(r.group(1))
            d["MAE"] = float(r.group(2))
    c = re.search(r"\[Check\].*->\s*(OK|MISMATCH)", log)
    if c:
        d["check"] = c.group(1)
    return d


## Run BACE, Clintox, ESOL and collect metrics

Captured stdout per task, then summary table.


In [3]:
rows = []
for task_name, summ_path in TASKS.items():
    summ_path = Path(summ_path)
    if not summ_path.is_file():
        print(f"[skip] missing {summ_path}")
        continue
    with open(summ_path, "r", encoding="utf-8") as f:
        summ = json.load(f)
    tt = summ["task_type"]
    paper = PAPER[task_name]

    print("=" * 76)
    print(f"TASK: {task_name}")
    print("=" * 76)
    log = run_eval(summ_path)
    print(log)

    parsed = parse_log(log, tt)
    row = {
        "task": task_name,
        "type": tt,
        "paper_table": paper["display"],
        "json_best_test": float(summ["best_test"]),
        "parity_check": parsed.get("check", ""),
    }
    if tt == "cls":
        row["repro_AUC"] = parsed.get("AUC")
        row["repro_F1_calib"] = parsed.get("F1_calib")
        if parsed.get("AUC") is not None:
            row["delta_vs_paper_AUC"] = round(parsed["AUC"] - paper["auc_0_1"], 6)
    else:
        row["repro_RMSE"] = parsed.get("RMSE")
        row["repro_MAE"] = parsed.get("MAE")
        if parsed.get("RMSE") is not None:
            row["delta_vs_paper_RMSE"] = round(parsed["RMSE"] - paper["rmse"], 6)
    rows.append(row)

summary_df = pd.DataFrame(rows)
print("\n### Summary table\n")
summary_df


TASK: bace


[Summary] /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/checkpoints/benchmark_bace/best_summary.json
[Ckpt]    /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/checkpoints/benchmark_bace/model.pth
[Reported] metric=AUC best_test=0.8932971014492753
[Device] cuda
[Preprocess] using artifact /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/checkpoints/benchmark_bace/eval_preprocess.pkl
[After filter] train=(artifact) valid=(skip) test=152
[Calib] calib.json  T=0.5 thr=0.44999999999999996

=== TEST (CLS) ===
AUC: 0.893297 | F1@calib_thr: 0.814371 | Coverage=0.4934, Acc@accepted=0.9067

[Check] reproduced=0.893297 vs reported=0.893297 -> OK

TASK: clintox


[Summary] /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/checkpoints/benchmark_clintox/best_summary.json
[Ckpt]    /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/checkpoints/benchmark_clintox/model.pth
[Reported] metric=AUC best_test=0.9496402877697843
[Device] cuda
[Preprocess] using artifact /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/checkpoints/benchmark_clintox/eval_preprocess.pkl
[After filter] train=(artifact) valid=(skip) test=148
[Calib] calib.json  T=0.5 thr=0.75

=== TEST (CLS) ===
AUC: 0.949640 | F1@calib_thr: 0.975089 | Coverage=0.9595, Acc@accepted=0.9648

[Check] reproduced=0.949640 vs reported=0.949640 -> OK

TASK: esol


[Summary] /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/checkpoints/benchmark_esol/best_summary.json
[Ckpt]    /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/checkpoints/benchmark_esol/model.pth
[Reported] metric=RMSE best_test=0.4790667260338801
[Device] cuda
[Preprocess] using artifact /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/checkpoints/benchmark_esol/eval_preprocess.pkl
[After filter] train=(artifact) valid=(skip) test=113

=== TEST (REG) ===
RMSE: 0.479067 | MAE: 0.356240

[Check] reproduced=0.479067 vs reported=0.479067 -> OK


### Summary table



,task,type,paper_table,json_best_test,parity_check,repro_AUC,repro_F1_calib,delta_vs_paper_AUC,repro_RMSE,repro_MAE,delta_vs_paper_RMSE
0,bace,cls,87.9 (AUC %),0.893297,OK,0.893297,0.814371,0.014297,NaN,NaN,NaN
1,clintox,cls,94.3 (AUC %),0.949640,OK,0.949640,0.975089,0.006640,NaN,NaN,NaN
2,esol,reg,0.517 (RMSE),0.479067,OK,NaN,NaN,NaN,0.479067,0.35624,-0.037933


## What to ship on GitHub

- `src/main_arcmol.py`, `src/attention_pooling_fusion.py`
- `checkpoints/benchmark_*` — `best_summary.json`, `model.pth`, `calib.json` (cls), `eval_preprocess.pkl`
- `data/<task>/{task}_test.pkl` only

**Optional:** with local `train`+`test` PKLs, run `export_eval_preprocess(CHECKPOINTS / "benchmark_bace" / "best_summary.json")` (per task) to refresh `eval_preprocess.pkl`.


### Optional: regenerate `eval_preprocess.pkl`

Uncomment when `data/<task>/` has both `*_train.pkl` and `*_test.pkl`:


In [4]:
# export_eval_preprocess(CHECKPOINTS / "benchmark_bace" / "best_summary.json")
# export_eval_preprocess(CHECKPOINTS / "benchmark_clintox" / "best_summary.json")
# export_eval_preprocess(CHECKPOINTS / "benchmark_esol" / "best_summary.json")
